In [1]:
import csv
import os
from datetime import datetime, date, timedelta

#  CONSTANTS
EQUIPMENT_FILE = "equipment.csv"
ALERT_FILE = "equipment_alerts.csv"

# The 14 equipment variables (Availability is calculated, never typed in).
FIELDS = [
    "Equipment ID", "Equipment Type", "Manufacturer", "Operating Hours",
    "Downtime", "Availability", "Temperature", "Vibration",
    "Fuel Consumption", "Brake Status", "Tyre Status", "Engine Status",
    "Maintenance Status", "Next Maintenance Date",
]
ALERT_FIELDS = ["Equipment ID", "Parameter", "Reading", "Status",
                "Recommended Action", "Date and Time"]

EQUIPMENT_TYPES = ["Haul Truck", "Loader", "Excavator", "Drilling Machine",
                   "Bulldozer", "Scraper Winch", "Crusher", "Conveyor"]
COMPONENT_STATUSES = ["Good", "Fair", "Poor"]
MAINTENANCE_STATUSES = ["Operational", "Under Maintenance", "Due", "Overdue"]

# Numeric fields and their rules (name: (minimum, maximum or None))
NUMERIC_FIELDS = {
    "Operating Hours": (0, None), "Downtime": (0, None),
    "Temperature": (-50, 300), "Vibration": (0, 100),
    "Fuel Consumption": (0, None),
}

# Thresholds (educational prototype values only)
TEMP_WARNING =80            # degrees C
TEMP_CRITICAL =100          # degrees C
VIB_WARNING 5                  # mm/s
VIB_CRITICAL = 8               # mm/s
MINIMUM_AVAILABILITY = 85                      # %
MAXIMUM_DOWNTIME = 20                          # hours
REPEATED_ALERT_LIMIT = 3                       # alerts
DUE_WINDOW_DAYS = 7                            # "due" if next maintenance within 7 days
DATE_FORMAT = "%Y-%m-%d"


# DATABASE FUNCTIONS

def load_equipment_data(file_name=EQUIPMENT_FILE):
    """Load equipment rows from the CSV file as a list of dictionaries."""
    # Start with an empty list
    equipment = []
    # If the file exists, read it row by row
    if os.path.exists(file_name):
        with open(file_name, "r", newline="") as f:
            equipment = list(csv.DictReader(f))
    else:
        # If the file does not exist, create an empty one
        save_equipment_data(equipment, file_name)
    # Return the list of equipment records
    return equipment


def save_equipment_data(equipment, file_name=EQUIPMENT_FILE):
    """Write the equipment list back to the CSV file."""
    with open(file_name, "w", newline="") as f:
        # Use the FIELDS list as the column headers
        writer = csv.DictWriter(f, fieldnames=FIELDS, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(equipment)


def equipment_id_exists(equipment, equipment_id):
    """Return True if the given Equipment ID is already in the list."""
    return any(item["Equipment ID"].lower() == equipment_id.strip().lower()
               for item in equipment)


def find_equipment(equipment, equipment_id):
    """Return the equipment record matching the ID, or None if not found."""
    for item in equipment:
        if item["Equipment ID"].lower() == equipment_id.strip().lower():
            return item
    return None


def calculate_availability(operating_time, downtime):
    """Availability (%) = Operating / (Operating + Downtime) * 100."""
    # Total time is the sum of operating time and downtime
    total = operating_time + downtime
    # Avoid division by zero
    if total == 0:
        return 0.0
    # Return the availability rounded to two decimal places
    return round(operating_time / total * 100, 2)


def validate_equipment(record):
    """Return a list of error messages (empty list means the record is valid)."""
    errors = []

    # 1. Check that no field (except Availability) is empty
    for field in FIELDS:
        if field == "Availability":
            continue
        if str(record.get(field, "")).strip() == "":
            errors.append(f"{field} cannot be empty.")
    if errors:
        return errors

    # 2. Check that numeric fields are numbers and within range
    for field, (low, high) in NUMERIC_FIELDS.items():
        try:
            value = float(record[field])
        except ValueError:
            errors.append(f"{field} must be a number.")
            continue
        if value < low or (high is not None and value > high):
            errors.append(f"{field} is out of range.")

    # 3. Check that categorical fields use allowed values
    if record["Equipment Type"] not in EQUIPMENT_TYPES:
        errors.append("Equipment Type must be one of: " + ", ".join(EQUIPMENT_TYPES))
    for field in ("Brake Status", "Tyre Status", "Engine Status"):
        if record[field] not in COMPONENT_STATUSES:
            errors.append(f"{field} must be one of: " + ", ".join(COMPONENT_STATUSES))
    if record["Maintenance Status"] not in MAINTENANCE_STATUSES:
        errors.append("Maintenance Status must be one of: " + ", ".join(MAINTENANCE_STATUSES))

    # 4. Check that the date is in the correct format
    try:
        datetime.strptime(record["Next Maintenance Date"], DATE_FORMAT)
    except ValueError:
        errors.append("Next Maintenance Date must be in YYYY-MM-DD format.")

    # Return the list of errors (empty if none)
    return errors


def add_equipment(equipment, record):
    """Validate and add one record. Returns (success, message)."""
    # Check that the Equipment ID is not empty
    if str(record.get("Equipment ID", "")).strip() == "":
        return False, "Equipment ID cannot be empty."
    # Check that the Equipment ID is unique
    if equipment_id_exists(equipment, record["Equipment ID"]):
        return False, "Equipment ID already exists."
    # Validate the rest of the record
    errors = validate_equipment(record)
    if errors:
        return False, " ".join(errors)

    # Clean up the record and calculate Availability
    record = {k: str(record[k]).strip() for k in FIELDS if k in record}
    record["Availability"] = calculate_availability(
        float(record["Operating Hours"]), float(record["Downtime"]))

    # Add to the list and save to file
    equipment.append(record)
    save_equipment_data(equipment)
    return True, "Equipment added successfully."


def search_equipment(equipment, keyword):
    """Keyword search across every field (matches Equipment ID too)."""
    keyword = keyword.strip().lower()
    # Return any record where the keyword appears in any field
    return [item for item in equipment
            if any(keyword in str(item.get(f, "")).lower() for f in FIELDS)]


def filter_equipment(equipment, field, value):
    """Exact (case-insensitive) match on one field."""
    if field not in FIELDS:
        return []
    return [item for item in equipment
            if str(item[field]).strip().lower() == value.strip().lower()]


# CONDITION MONITORING FUNCTIONS

def check_temperature(temperature):
    """Return 'CRITICAL', 'WARNING', or 'NORMAL' for a temperature reading."""
    if temperature > TEMP_CRITICAL:
        return "CRITICAL"
    if temperature >= TEMP_WARNING:
        return "WARNING"
    return "NORMAL"


def check_vibration(vibration):
    """Return 'CRITICAL', 'WARNING', or 'NORMAL' for a vibration reading."""
    if vibration > VIB_CRITICAL:
        return "CRITICAL"
    if vibration >= VIB_WARNING:
        return "WARNING"
    return "NORMAL"


def get_overall_status(*statuses):
    """Combine several statuses into one overall status (worst wins)."""
    if "CRITICAL" in statuses:
        return "CRITICAL"
    if "WARNING" in statuses:
        return "WARNING"
    return "NORMAL"


def create_alert(equipment_id, parameter, value, status, unit=""):
    """Return an alert dictionary, or None when status is NORMAL."""
    if status == "NORMAL":
        return None
    # Choose the recommended action based on severity
    if status == "CRITICAL":
        action = "Stop equipment and inspect immediately."
    else:
        action = "Monitor equipment and schedule maintenance."
    # Build and return the alert dictionary
    return {
        "Equipment ID": equipment_id,
        "Parameter": parameter,
        "Reading": f"{value}{unit}",
        "Status": status,
        "Recommended Action": action,
        "Date and Time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    }


def format_alert(alert):
    """Return a text block in the style of Figure 1 of the project brief."""
    bar = "=" * 53
    return "\n".join([
        bar, "                MINING MONITORING ALERT", bar,
        f"                      [{alert['Status']}]",
        f"Equipment ID: {alert['Equipment ID']}",
        f"Parameter: {alert['Parameter']}",
        f"Value: {alert['Reading']}",
        "Status: " + ("Above Critical Threshold" if alert["Status"] == "CRITICAL"
                      else "Within Warning Range"),
        "Recommended Action:", f"    {alert['Recommended Action']}", bar,
    ])


def save_alert(alert, file_name=ALERT_FILE):
    """Append an alert to the alerts CSV file."""
    if alert is None:
        return
    new_file = not os.path.exists(file_name)
    with open(file_name, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=ALERT_FIELDS)
        if new_file:
            writer.writeheader()
        writer.writerow(alert)


def load_alerts(file_name=ALERT_FILE):
    """Load all alerts from the alerts CSV file as a list of dictionaries."""
    if not os.path.exists(file_name):
        return []
    with open(file_name, "r", newline="") as f:
        return list(csv.DictReader(f))


def count_alerts(equipment_id, alerts=None):
    """Return the number of alerts for a given Equipment ID."""
    if alerts is None:
        alerts = load_alerts()
    return sum(1 for a in alerts if a["Equipment ID"].lower() == equipment_id.lower())


def monitor_equipment(item, save=True):
    """
    Check one equipment record using its stored Temperature and Vibration.
    Returns (overall_status, list_of_alerts).
    """
    # Read the stored sensor values
    temperature = float(item["Temperature"])
    vibration = float(item["Vibration"])

    # Check each reading against the thresholds
    t_status = check_temperature(temperature)
    v_status = check_vibration(vibration)

    # Create alerts only for WARNING or CRITICAL readings
    alerts = [a for a in (
        create_alert(item["Equipment ID"], "Temperature", temperature, t_status, " °C"),
        create_alert(item["Equipment ID"], "Vibration", vibration, v_status, " mm/s"),
    ) if a is not None]

    # Save the alerts to file
    if save:
        for a in alerts:
            save_alert(a)

    # Return the combined status and the list of alerts
    return get_overall_status(t_status, v_status), alerts


def update_readings(equipment, equipment_id, temperature, vibration):
    """Store new sensor readings in the database. Returns (success, message)."""
    # Find the equipment record
    item = find_equipment(equipment, equipment_id)
    if item is None:
        return False, "Equipment ID not found."

    # Convert the readings to numbers
    try:
        t, v = float(temperature), float(vibration)
    except ValueError:
        return False, "Temperature and vibration must be numbers."

    # Check that the readings are within allowed ranges
    if not (NUMERIC_FIELDS["Temperature"][0] <= t <= NUMERIC_FIELDS["Temperature"][1]) or \
       not (NUMERIC_FIELDS["Vibration"][0] <= v <= NUMERIC_FIELDS["Vibration"][1]):
        return False, "Reading is out of range."

    # Store the new readings and save the file
    item["Temperature"], item["Vibration"] = str(t), str(v)
    save_equipment_data(equipment)
    return True, "Readings updated."


# MAINTENANCE MONITORING FUNCTIONS

def maintenance_state(item, today=None):
    """
    Derive the real maintenance state from status + Next Maintenance Date.
    Returns one of: 'Under Maintenance', 'Overdue', 'Due', 'OK'.
    """
    today = today or date.today()
    status = item["Maintenance Status"].strip().lower()

    # If the status says 'under maintenance', return it immediately
    if status == "under maintenance":
        return "Under Maintenance"

    # Otherwise, compare the next maintenance date to today
    next_date = datetime.strptime(item["Next Maintenance Date"], DATE_FORMAT).date()
    if status == "overdue" or next_date < today:
        return "Overdue"
    if status == "due" or next_date <= today + timedelta(days=DUE_WINDOW_DAYS):
        return "Due"
    return "OK"


def check_maintenance_problems(item, alert_count, today=None):
    """Return (problems list, availability) for one equipment record."""
    problems = []

    # Calculate availability from operating hours and downtime
    operating = float(item["Operating Hours"])
    downtime = float(item["Downtime"])
    availability = calculate_availability(operating, downtime)

    # Check availability against the minimum
    if availability < MINIMUM_AVAILABILITY:
        problems.append("Low Availability")

    # Check the maintenance state
    state = maintenance_state(item, today)
    if state == "Overdue":
        problems.append("Overdue Maintenance")
    elif state == "Due":
        problems.append("Due for Maintenance")
    elif state == "Under Maintenance":
        problems.append("Under Maintenance")

    # Check downtime against the maximum
    if downtime > MAXIMUM_DOWNTIME:
        problems.append("Excessive Downtime")

    # Check for too many repeated alerts
    if alert_count >= REPEATED_ALERT_LIMIT:
        problems.append("Repeated Alerts")

    # Return the list of problems and the availability
    return problems, availability


def create_maintenance_priority_list(all_equipment, alerts=None, today=None):
    """
    Equipment with problems, most urgent first: most problems, then lowest
    availability (so low-availability machines are prioritised on ties).
    """
    if alerts is None:
        alerts = load_alerts()

    priority = []
    for item in all_equipment:
        # Count the alerts for this equipment
        count = count_alerts(item["Equipment ID"], alerts)
        # Check the maintenance problems
        problems, availability = check_maintenance_problems(item, count, today)
        # Only include equipment that has at least one problem
        if problems:
            priority.append({
                "Equipment ID": item["Equipment ID"],
                "Problems": problems,
                "Availability": availability,
                "Number of Problems": len(problems),
            })

    # Sort by most problems first, then by lowest availability
    priority.sort(key=lambda p: (-p["Number of Problems"], p["Availability"]))
    return priority


def generate_maintenance_alerts(all_equipment, today=None, save=True):
    """Alerts required by brief 3.h: overdue maintenance and excessive downtime."""
    alerts = []
    for item in all_equipment:
        # Alert if maintenance is overdue
        if maintenance_state(item, today) == "Overdue":
            alerts.append(create_alert(item["Equipment ID"], "Maintenance", "Overdue", "WARNING"))
        # Alert if downtime is excessive
        if float(item["Downtime"]) > MAXIMUM_DOWNTIME:
            alerts.append(create_alert(item["Equipment ID"], "Downtime",
                                       f"{item['Downtime']} h", "WARNING"))
    # Save the alerts if requested
    if save:
        for a in alerts:
            save_alert(a)
    return alerts


def equipment_summary(all_equipment, alerts=None):
    """KPI numbers for Student D's dashboard."""
    if alerts is None:
        alerts = load_alerts()

    n = len(all_equipment)

    # Calculate availability for every item
    avail = [calculate_availability(float(i["Operating Hours"]), float(i["Downtime"]))
             for i in all_equipment]

    # Calculate the overall status for every item
    statuses = [get_overall_status(check_temperature(float(i["Temperature"])),
                                   check_vibration(float(i["Vibration"])))
                for i in all_equipment]

    # Return the KPI dictionary
    return {
        "Total equipment": n,
        "Under maintenance": sum(1 for i in all_equipment
                                 if maintenance_state(i) == "Under Maintenance"),
        "Average availability (%)": round(sum(avail) / n, 2) if n else 0,
        "Equipment operating normally (%)": round(statuses.count("NORMAL") / n * 100, 1) if n else 0,
        "Critical equipment": statuses.count("CRITICAL"),
        "Total equipment alerts": len(alerts),
    }


# ====================================================================
# CONSOLE INTERFACE (thin wrapper for testing / demo)
# ====================================================================

def print_equipment(items):
    """Print every field of every equipment record."""
    if not items:
        print("No equipment found.")
        return
    for item in items:
        print()
        for field in FIELDS:
            print(f"{field}: {item[field]}")


def add_equipment_console(equipment):
    """Ask the user for each field and add a new equipment record."""
    record = {}
    for field in FIELDS:
        if field == "Availability":
            continue
        record[field] = input(f"Enter {field}: ").strip()
        # Check for duplicate ID early
        if field == "Equipment ID" and equipment_id_exists(equipment, record[field]):
            print("Error: Equipment ID already exists.")
            return
    # Validate and add the record
    ok, msg = add_equipment(equipment, record)
    print(msg if ok else "Error: " + msg)


def equipment_database(equipment):
    """Menu for the equipment database."""
    while True:
        print("\nEquipment Database\n1. Add\n2. View\n3. Search\n4. Filter\n5. Exit")
        choice = input("Choose an option: ").strip()
        if choice == "1":
            add_equipment_console(equipment)
        elif choice == "2":
            print_equipment(equipment)
        elif choice == "3":
            print_equipment(search_equipment(equipment, input("Equipment ID or keyword: ")))
        elif choice == "4":
            # Let the user choose which field to filter by
            for n, f in enumerate(FIELDS, 1):
                print(f"{n} - {f}")
            try:
                field = FIELDS[int(input("Field number: ")) - 1]
            except (ValueError, IndexError):
                print("Invalid field.")
                continue
            print_equipment(filter_equipment(equipment, field, input("Value: ")))
        elif choice == "5":
            break
        else:
            print("Invalid option. Please choose 1-5.")


def condition_monitoring(equipment):
    """Menu for condition monitoring."""
    while True:
        print("\nEquipment Condition Monitoring\n1. Monitor one item\n2. Update readings\n3. Exit")
        choice = input("Choose an option: ").strip()
        if choice == "1":
            item = find_equipment(equipment, input("Equipment ID: "))
            if item is None:
                print("Equipment ID not found.")
                continue
            status, alerts = monitor_equipment(item)
            for a in alerts:
                print(format_alert(a))
            print("Overall status:", status)
        elif choice == "2":
            ok, msg = update_readings(equipment, input("Equipment ID: "),
                                      input("Temperature (°C): "), input("Vibration (mm/s): "))
            print(msg)
        elif choice == "3":
            break
        else:
            print("Invalid option. Please choose 1-3.")


def maintenance_monitoring(equipment):
    """Menu for maintenance monitoring."""
    while True:
        print("\nEquipment Maintenance Monitoring\n1. View priority list\n2. Exit")
        choice = input("Choose an option: ").strip()
        if choice == "1":
            priority = create_maintenance_priority_list(equipment)
            if not priority:
                print("No equipment requires maintenance.")
            for pos, p in enumerate(priority, 1):
                print(f"\nPriority {pos}: {p['Equipment ID']}  "
                      f"Availability {p['Availability']}%  Problems: {p['Number of Problems']}")
                for problem in p["Problems"]:
                    print(" -", problem)
        elif choice == "2":
            break
        else:
            print("Invalid option. Please choose 1 or 2.")


def main():
    """Main menu — the entry point of the program."""
    equipment = load_equipment_data()
    while True:
        print("\nStudent B - Equipment Module\n1. Equipment Database\n"
              "2. Condition Monitoring\n3. Maintenance Monitoring\n4. Exit")
        choice = input("Choose an option: ").strip()
        if choice == "1":
            equipment_database(equipment)
        elif choice == "2":
            condition_monitoring(equipment)
        elif choice == "3":
            maintenance_monitoring(equipment)
        elif choice == "4":
            break
        else:
            print("Invalid option. Please choose 1-4.")


if __name__ == "__main__":
    main()



Student B - Equipment Module
1. Equipment Database
2. Condition Monitoring
3. Maintenance Monitoring
4. Exit


Choose an option:  2



Equipment Condition Monitoring
1. Monitor one item
2. Update readings
3. Exit


Choose an option:  1
Equipment ID:  HAUL TRUCK


Equipment ID not found.

Equipment Condition Monitoring
1. Monitor one item
2. Update readings
3. Exit


Choose an option:  3



Student B - Equipment Module
1. Equipment Database
2. Condition Monitoring
3. Maintenance Monitoring
4. Exit


Choose an option:  4
